# Big Data Modelling and Management - Lab 1

**Prepare for the class:**

0. download Dataset from Moodle Lab 1 -> upzip the folder + follow installation instruction file
1. run Jupyter 
2. run Neo4j from cmd or as Windows service
3. open http://localhost:7474/browser/  
4. log-in with login: neo4j, password: [yourpassword]
5. start exploring the database

My email for questions: osavchuk@novaims.unl.pt 

## Graphs and Graph databases

![neo4j_example](img/neo4j_logo.png)  

## Data Model of graph database

**Data Model**

The data model describes the labels of nodes, relationships, and properties for the graph. It does not have specific data that will be created in the graph.

A graph data model, however is important because it defines the names that will be used for labels, relationship types, and properties when the graph is created and used by the application.

![sample-data-model.png](img/sample-data-model.png)

## Cypher Language

**Node and Relationship**  
A node can be represented with:
```
()                       //anonymous node (no label or variable) can refer to any node in the database
(p:Person)               //using variable p and node Person
(:Movie)                 //no variable, label Movie
(film:Movie)             //using variable film and label Movie
(p:Person{name: "Jonh"}) //node Person with property name
```
And relationships as:
```
()-[:ACTED_IN]-()                    // Relationship of Label type ACTED_IN between two anonymous nodes
(:Movie)<-[:ACTED_IN]-(p:Person)     // Unidirectional relationship between Movie and Person 
()-[h:HAS]-()                        // Relationship with variable h and label Has
```

Would be represented as:  
```
(:Person{name: "John Costa"})-[:ACTED_IN]->(:Movie{title: "Movie XYZ"})
```

## Hands on!

For this class, we will continue to use docker with a neo4j database.

There are several libraries to interact with neo4j with python. We will use python's neo4j library as it is the simplest to use. There are plenty more.  
Another way to interact with neo4j is to use the web interface available at: http://localhost:7474/browser/ (pretty graphs there)
```
Connect URL : bolt://http://localhost:7474
Username: neo4j 
Password: neo4j (first time, later - as you have set)  
```

Ok, let's start!

In [39]:
! pip install neo4j

In [49]:
from neo4j import GraphDatabase
from pprint import pprint

## Define a Connection to your external Database

In [51]:
NEO4J_URI="neo4j://localhost:7687"
NEO4J_USERNAME="neo4j"
NEO4J_PASSWORD="Neo4jneo4j" # add your password here!

In [42]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD), )

In [43]:
driver.verify_connectivity()
print("Connected to Neo4j!")

Connected to Neo4j!


## All The Functions you'll need to run queries in Neo4J

In [44]:
def execute_query_deprecated(driver, query):
    return list(driver.execute_query(query))

In [45]:
def execute_write(driver, query):
    with driver.session(database="neo4j") as session:
        # Write transactions allow the driver to handle retries and transient errors
        result = session.execute_write(lambda tx, query: list(tx.run(query)), query)
    return result

In [46]:
def execute_read(driver, query):    
    with driver.session(database="neo4j") as session:
        result = session.execute_read(lambda tx, query: list(tx.run(query)), query)
    return result

## Explore database

In [52]:
# here we use Cypher clause 'CALL' and built-in procedure 'db.labels'

query = """
       CALL db.labels();
    """

result = execute_read(driver, query)

pprint(result)

[<Record label='Movie'>, <Record label='Person'>]


In [47]:
query = """
        CALL db.relationshipTypes();
    """

result = execute_read(driver, query)

pprint(result)

[<Record relationshipType='ACTED_IN'>,
 <Record relationshipType='DIRECTED'>,
 <Record relationshipType='PRODUCED'>,
 <Record relationshipType='WROTE'>,
 <Record relationshipType='FOLLOWS'>,
 <Record relationshipType='REVIEWED'>]


## Query Tools

### Match Operator

Equivalent to SQL `select` fetchs data based on query specifications. Lets see some examples:

#### Get all nodes in the database, no constraints.

In [53]:
query = """
        MATCH (u)
        RETURN u
        LIMIT 3
    """

result = execute_read(driver, query)
pprint(result)

[<Record u=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:0' labels=frozenset({'Movie'}) properties={'tagline': 'Welcome to the Real World', 'title': 'The Matrix', 'released': 1999}>>,
 <Record u=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:1' labels=frozenset({'Person'}) properties={'born': 1964, 'name': 'Keanu Reeves'}>>,
 <Record u=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:2' labels=frozenset({'Person'}) properties={'born': 1967, 'name': 'Carrie-Anne Moss'}>>]


#### Get only Locations

In [54]:
query = """
        MATCH (p:Person)
        RETURN *
        LIMIT 3
    """

result = execute_read(driver, query)
pprint(result)

[<Record p=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:1' labels=frozenset({'Person'}) properties={'born': 1964, 'name': 'Keanu Reeves'}>>,
 <Record p=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:2' labels=frozenset({'Person'}) properties={'born': 1967, 'name': 'Carrie-Anne Moss'}>>,
 <Record p=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:3' labels=frozenset({'Person'}) properties={'born': 1961, 'name': 'Laurence Fishburne'}>>]


#### Get variable from Node

In [59]:
query = """
        MATCH (p:Person)
        RETURN p.name as name
        LIMIT 10
    """

result = execute_read(driver, query)
pprint(result)

[<Record name='Keanu Reeves'>,
 <Record name='Carrie-Anne Moss'>,
 <Record name='Laurence Fishburne'>,
 <Record name='Hugo Weaving'>,
 <Record name='Lilly Wachowski'>,
 <Record name='Lana Wachowski'>,
 <Record name='Joel Silver'>,
 <Record name='Emil Eifrem'>,
 <Record name='Charlize Theron'>,
 <Record name='Al Pacino'>]


#### Conditional match in 2 different ways

Passing the property on the node

In [62]:
# WHERE clause

query = """
        MATCH (p:Person)
        WHERE p.name='Al Pacino'
        RETURN p
    """

result = execute_read(driver, query)
pprint(result)

[<Record p=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:13' labels=frozenset({'Person'}) properties={'born': 1940, 'name': 'Al Pacino'}>>]


In [61]:
query = """
        MATCH (p:Person {name: 'Al Pacino'})
        RETURN p.name as Actor
        
    """

result = execute_read(driver, query)

pprint(result)

[<Record Actor='Al Pacino'>]


In [80]:
# WHERE clause for finding specific node by ID

query = """
        MATCH (p:Person)
        WHERE elementId(p) = '4:1a3f2155-98eb-46c3-971f-bf01de862908:13'
        RETURN p
    """

result = execute_read(driver, query)
pprint(result) 

[<Record p=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:13' labels=frozenset({'Person'}) properties={'born': 1940, 'name': 'Al Pacino'}>>]


### Building a path

In [93]:
query = """
        MATCH path = ()-[]->()-->()
        RETURN path 
        LIMIT 5
    """
result = execute_read(driver, query)
pprint(result)

[<Record path=<Path start=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:167' labels=frozenset({'Person'}) properties={'name': 'Paul Blythe'}> end=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:169' labels=frozenset({'Person'}) properties={'name': 'Jessica Thompson'}> size=2>>,
 <Record path=<Path start=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:168' labels=frozenset({'Person'}) properties={'name': 'Angela Scope'}> end=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:105' labels=frozenset({'Movie'}) properties={'tagline': 'Everything is connected', 'title': 'Cloud Atlas', 'released': 2012}> size=2>>,
 <Record path=<Path start=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:170' labels=frozenset({'Person'}) properties={'name': 'James Thompson'}> end=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:105' labels=frozenset({'Movie'}) properties={'tagline': 'Everything is connected', 'title': 'Cloud Atlas', 'released': 2012}> size=2>

In [94]:
query = """
        MATCH path = (:Person)-[]->(:Movie)
        RETURN path 
        SKIP 46
        LIMIT 5
    """
result = execute_read(driver, query)
pprint(result) 

[<Record path=<Path start=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:34' labels=frozenset({'Person'}) properties={'born': 1961, 'name': 'Meg Ryan'}> end=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:29' labels=frozenset({'Movie'}) properties={'tagline': 'I feel the need, the need for speed.', 'title': 'Top Gun', 'released': 1986}> size=1>>,
 <Record path=<Path start=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:33' labels=frozenset({'Person'}) properties={'born': 1933, 'name': 'Tom Skerritt'}> end=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:29' labels=frozenset({'Movie'}) properties={'tagline': 'I feel the need, the need for speed.', 'title': 'Top Gun', 'released': 1986}> size=1>>,
 <Record path=<Path start=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:169' labels=frozenset({'Person'}) properties={'name': 'Jessica Thompson'}> end=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:37' labels=frozenset({'Movie'}) propertie

In [91]:
query = """
        MATCH path = (p:Person {name: "Tom Hanks"})-[]->(m:Movie)
        RETURN path 
        LIMIT 5
    """
result = execute_read(driver, query)
pprint(result)

[<Record path=<Path start=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:68' labels=frozenset({'Person'}) properties={'born': 1956, 'name': 'Tom Hanks'}> end=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:162' labels=frozenset({'Movie'}) properties={'tagline': 'Once in a lifetime you get a chance to do something different.', 'title': 'A League of Their Own', 'released': 1992}> size=1>>,
 <Record path=<Path start=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:68' labels=frozenset({'Person'}) properties={'born': 1956, 'name': 'Tom Hanks'}> end=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:161' labels=frozenset({'Movie'}) properties={'tagline': 'This Holiday Season... Believe', 'title': 'The Polar Express', 'released': 2004}> size=1>>,
 <Record path=<Path start=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:68' labels=frozenset({'Person'}) properties={'born': 1956, 'name': 'Tom Hanks'}> end=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01

In [83]:
query = """
        MATCH path = (p:Person {name: "Tom Hanks"})-[]->()
        RETURN path 
        LIMIT 5
    """
result = execute_read(driver, query)
pprint(result) 

[<Record path=<Path start=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:68' labels=frozenset({'Person'}) properties={'born': 1956, 'name': 'Tom Hanks'}> end=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:162' labels=frozenset({'Movie'}) properties={'tagline': 'Once in a lifetime you get a chance to do something different.', 'title': 'A League of Their Own', 'released': 1992}> size=1>>,
 <Record path=<Path start=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:68' labels=frozenset({'Person'}) properties={'born': 1956, 'name': 'Tom Hanks'}> end=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:161' labels=frozenset({'Movie'}) properties={'tagline': 'This Holiday Season... Believe', 'title': 'The Polar Express', 'released': 2004}> size=1>>,
 <Record path=<Path start=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:68' labels=frozenset({'Person'}) properties={'born': 1956, 'name': 'Tom Hanks'}> end=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01

In [79]:
query = """
        MATCH (p:Person {name: "Tom Hanks"})-[:ACTED_IN]->(m:Movie)
        RETURN m.title
    """
result = execute_read(driver, query)
pprint(result) 

[<Record m.title='A League of Their Own'>,
 <Record m.title='The Polar Express'>,
 <Record m.title="Charlie Wilson's War">,
 <Record m.title='Cast Away'>,
 <Record m.title='Apollo 13'>,
 <Record m.title='The Green Mile'>,
 <Record m.title='The Da Vinci Code'>,
 <Record m.title='Cloud Atlas'>,
 <Record m.title='That Thing You Do'>,
 <Record m.title='Joe Versus the Volcano'>,
 <Record m.title='Sleepless in Seattle'>,
 <Record m.title="You've Got Mail">]


### Counting and grouping

In [95]:
query = """
        MATCH path = (:Person {name: 'James Thompson'})-[r:REVIEWED]->()
        RETURN count(r) as Sum 
        LIMIT 5
    """
result = execute_read(driver, query)
pprint(result)

[<Record Sum=2>]


In [101]:
query = """
        MATCH path = (:Person)-[r:REVIEWED]-(:Movie)
        RETURN count(*) as Sum 
        LIMIT 5
    """
result = execute_read(driver, query)
pprint(result)

[<Record Sum=9>]


In [109]:
query = """
        MATCH (m:Movie)
        RETURN m.released, COUNT(*) as count
        ORDER by count desc // by m.released or by m.title (error)
        LIMIT 15
    """

result = execute_read(driver, query)
pprint(result)

[<Record m.released=1999 count=4>,
 <Record m.released=1992 count=4>,
 <Record m.released=2006 count=3>,
 <Record m.released=1998 count=3>,
 <Record m.released=2003 count=3>,
 <Record m.released=1996 count=3>,
 <Record m.released=2000 count=3>,
 <Record m.released=2008 count=2>,
 <Record m.released=1995 count=2>,
 <Record m.released=1997 count=2>,
 <Record m.released=1986 count=2>,
 <Record m.released=2009 count=1>,
 <Record m.released=2012 count=1>,
 <Record m.released=1990 count=1>,
 <Record m.released=1993 count=1>]


In [110]:
query = """
        MATCH (node)
        RETURN COUNT(*) as count
    """

result = execute_read(driver, query)
pprint(result)

[<Record count=171>]


In [112]:
query = """
        MATCH (p:Person)-[r:REVIEWED]-()
        RETURN p, avg(r.rating) AS AverageRating
    """

result = execute_read(driver, query)
pprint(result)

[<Record p=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:169' labels=frozenset({'Person'}) properties={'name': 'Jessica Thompson'}> AverageRating=75.0>,
 <Record p=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:170' labels=frozenset({'Person'}) properties={'name': 'James Thompson'}> AverageRating=82.5>,
 <Record p=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:168' labels=frozenset({'Person'}) properties={'name': 'Angela Scope'}> AverageRating=62.0>]


#### The use of **f-strings** and using two way paths

In [115]:
actorName = 'Tom Hanks'

query = f"""
        MATCH (:Person {{name: '{actorName}'}})-[:ACTED_IN]->(m)<-[:ACTED_IN]-(coActors)
        RETURN coActors.name
    """

result = execute_read(driver, query)
pprint(result)

[<Record coActors.name='Geena Davis'>,
 <Record coActors.name='Bill Paxton'>,
 <Record coActors.name='Madonna'>,
 <Record coActors.name='Lori Petty'>,
 <Record coActors.name="Rosie O'Donnell">,
 <Record coActors.name='Philip Seymour Hoffman'>,
 <Record coActors.name='Julia Roberts'>,
 <Record coActors.name='Helen Hunt'>,
 <Record coActors.name='Ed Harris'>,
 <Record coActors.name='Gary Sinise'>,
 <Record coActors.name='Bill Paxton'>,
 <Record coActors.name='Kevin Bacon'>,
 <Record coActors.name='Sam Rockwell'>,
 <Record coActors.name='James Cromwell'>,
 <Record coActors.name='David Morse'>,
 <Record coActors.name='Gary Sinise'>,
 <Record coActors.name='Patricia Clarkson'>,
 <Record coActors.name='Bonnie Hunt'>,
 <Record coActors.name='Michael Clarke Duncan'>,
 <Record coActors.name='Paul Bettany'>,
 <Record coActors.name='Ian McKellen'>,
 <Record coActors.name='Audrey Tautou'>,
 <Record coActors.name='Halle Berry'>,
 <Record coActors.name='Hugo Weaving'>,
 <Record coActors.name='Jim Br

In [119]:
actorName = 'Tom Hanks'

query = f"""
        MATCH path= (:Person  {{name: '{actorName}'}})-[:ACTED_IN]->(m)<-[:ACTED_IN]-(coActors)
        RETURN path
        LIMIT 10
    """

result = execute_read(driver, query)
pprint(result)

[<Record path=<Path start=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:68' labels=frozenset({'Person'}) properties={'born': 1956, 'name': 'Tom Hanks'}> end=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:164' labels=frozenset({'Person'}) properties={'born': 1956, 'name': 'Geena Davis'}> size=2>>,
 <Record path=<Path start=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:68' labels=frozenset({'Person'}) properties={'born': 1956, 'name': 'Tom Hanks'}> end=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:146' labels=frozenset({'Person'}) properties={'born': 1955, 'name': 'Bill Paxton'}> size=2>>,
 <Record path=<Path start=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:68' labels=frozenset({'Person'}) properties={'born': 1956, 'name': 'Tom Hanks'}> end=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:163' labels=frozenset({'Person'}) properties={'born': 1954, 'name': 'Madonna'}> size=2>>,
 <Record path=<Path start=<Node element_id='4:1

In [128]:
query = """MATCH  path = (:Person {name: 'Tom Hanks'})-[:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(coActors:Person),
                  directorPath = (:Person {name: 'Ron Howard'})-[:DIRECTED]->(m)
            
           RETURN path, directorPath
    """

result = execute_read(driver, query)
pprint(result)

[<Record path=<Path start=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:68' labels=frozenset({'Person'}) properties={'born': 1956, 'name': 'Tom Hanks'}> end=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:145' labels=frozenset({'Person'}) properties={'born': 1950, 'name': 'Ed Harris'}> size=2> directorPath=<Path start=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:115' labels=frozenset({'Person'}) properties={'born': 1954, 'name': 'Ron Howard'}> end=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:144' labels=frozenset({'Movie'}) properties={'tagline': 'Houston, we have a problem.', 'title': 'Apollo 13', 'released': 1995}> size=1>>,
 <Record path=<Path start=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:68' labels=frozenset({'Person'}) properties={'born': 1956, 'name': 'Tom Hanks'}> end=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:134' labels=frozenset({'Person'}) properties={'born': 1955, 'name': 'Gary Sinise'}> size=2> dire

In [131]:
query = """
MATCH (actor:Person)-[:ACTED_IN]->(m:Movie),
      (director:Person)-[:DIRECTED]->(m),
      (producer:Person)-[:PRODUCED]->(m),
      (reviewer:Person)-[:REVIEWED]->(m)
RETURN actor, m, director, producer, reviewer     
    """

result = execute_read(driver, query)
pprint(result)

[<Record actor=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:106' labels=frozenset({'Person'}) properties={'born': 1966, 'name': 'Halle Berry'}> m=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:105' labels=frozenset({'Movie'}) properties={'tagline': 'Everything is connected', 'title': 'Cloud Atlas', 'released': 2012}> director=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:6' labels=frozenset({'Person'}) properties={'born': 1965, 'name': 'Lana Wachowski'}> producer=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:110' labels=frozenset({'Person'}) properties={'born': 1961, 'name': 'Stefan Arndt'}> reviewer=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:169' labels=frozenset({'Person'}) properties={'name': 'Jessica Thompson'}>>,
 <Record actor=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:68' labels=frozenset({'Person'}) properties={'born': 1956, 'name': 'Tom Hanks'}> m=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:10

In [132]:
query = """
MATCH (p:Person)-[r:ACTED_IN|DIRECTED|PRODUCED|REVIEWED]->(m:Movie)
RETURN p.name AS Person, type(r) AS Role, m.title AS Movie    
LIMIT 10
    """

result = execute_read(driver, query)
pprint(result)

[<Record Person='Emil Eifrem' Role='ACTED_IN' Movie='The Matrix'>,
 <Record Person='Joel Silver' Role='PRODUCED' Movie='The Matrix'>,
 <Record Person='Lana Wachowski' Role='DIRECTED' Movie='The Matrix'>,
 <Record Person='Lilly Wachowski' Role='DIRECTED' Movie='The Matrix'>,
 <Record Person='Hugo Weaving' Role='ACTED_IN' Movie='The Matrix'>,
 <Record Person='Laurence Fishburne' Role='ACTED_IN' Movie='The Matrix'>,
 <Record Person='Carrie-Anne Moss' Role='ACTED_IN' Movie='The Matrix'>,
 <Record Person='Keanu Reeves' Role='ACTED_IN' Movie='The Matrix'>,
 <Record Person='Joel Silver' Role='PRODUCED' Movie='The Matrix Reloaded'>,
 <Record Person='Lana Wachowski' Role='DIRECTED' Movie='The Matrix Reloaded'>]


## CRUD Operations

### CREATE

Each node is given a unique id implicitly

In [135]:
query = """
        CREATE (u1:User)
        RETURN u1
"""
result = execute_write(driver, query)
pprint(result)

[<Record u1=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:171' labels=frozenset({'User'}) properties={}>>]


In [138]:
query = """
        CREATE (u:User)-[:LEARNS]->(c:Course {name: 'BDMM'})
        RETURN *
"""
result = execute_write(driver, query)
pprint(result)

[<Record c=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:178' labels=frozenset({'Course'}) properties={'name': 'BDMM'}> u=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:177' labels=frozenset({'User'}) properties={}>>]


In [137]:
query = """
        CREATE (u1:User)-[:LEARNS]->(c:Course{name: "BDMM"})-[:MENTIONS]->(u2:User{name: "XYZ"})
        RETURN *
"""
result = execute_write(driver, query)
pprint(result)

[<Record c=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:175' labels=frozenset({'Course'}) properties={'name': 'BDMM'}> u1=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:174' labels=frozenset({'User'}) properties={}> u2=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:176' labels=frozenset({'User'}) properties={'name': 'XYZ'}>>]


In [139]:
# Create more than one label at once
query = """
        CREATE (N:User:Student {name: "Alice"})
        RETURN *
"""
result = execute_write(driver, query)
pprint(result)

[<Record N=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:179' labels=frozenset({'User', 'Student'}) properties={'name': 'Alice'}>>]


In [140]:
import numpy as np

user_1 = f"user-{np.random.randint(1000000)}"
user_2 = f"user-{np.random.randint(1000000)}"

tweet_content  = "Yet another tweet"

query = f"""
        CREATE (u1:User{{username: "{user_1}"}})-[:TWEETED]->\
              (t:Tweet{{text: "{tweet_content}"}})-[:MENTIONS]->\
              (u2:User{{username: "{user_2}"}})
        RETURN *
"""
result = execute_write(driver, query)
pprint(result)

[<Record t=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:181' labels=frozenset({'Tweet'}) properties={'text': 'Yet another tweet'}> u1=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:180' labels=frozenset({'User'}) properties={'username': 'user-51184'}> u2=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:182' labels=frozenset({'User'}) properties={'username': 'user-142075'}>>]


### SET

Create, if it exists update.   
Can be compared with an `UPSERT` which exists in some SQL languages like PostgresQL

In [141]:
user = user_1

query = f"""
        MATCH (u1:User{{username: \"{user}\"}})
        SET u1.name="joao" , u1.age = 34
        RETURN *
    """

result = execute_write(driver, query)

pprint(result)

[<Record u1=<Node element_id='4:1a3f2155-98eb-46c3-971f-bf01de862908:180' labels=frozenset({'User'}) properties={'name': 'joao', 'age': 34, 'username': 'user-51184'}>>]


### Delete
**Please be carefull when executing delete statements on any shared database**

In [142]:
query = """
      MATCH (u:User) 
      WHERE u.username =~ "user-.*"
      DETACH DELETE u
      RETURN count(u)
    """

result = execute_write(driver, query)

pprint(result)

[<Record count(u)=2>]


**Instance Model**

An important part of the graph data modeling process is to test the model against the use cases. To do this, you need to have a set of sample data that you can use to see if the use cases can be answered with the model.

In this instance model, we have created some instances of Person and Movie nodes, as well as their relationships. Having this type of instance model will help us to test our use cases.

![sample-data-instance-model.png](img/sample-data-instance-model.png)

**Fanout**

Fanout is when entities are represented not as a single node, but as a network or linked nodes. These additional nodes could have been represented as properties of the 'main' nodes - Person and Residence, respectively.

The main risk about fanout is that it can lead to very dense nodes, or supernodes. These are nodes that have hundreds of thousands of incoming or outgoing relationships Supernodes need to be handled carefully.

![neo4j_fanout.png](img/neo4j_fanout.png)

---
---
---
## What's next ?! 

The **first project will be released soon!** Doubts and questions please post on the Moodle Forum.



![feb14.png](img/feb14.png)

**COMPARISON OF SQL and NoSQL databases**

Graph databases are not the most common type of database as they are a completely different way of modeling data which is very intuitive for the data with graph-like structure.
In graph databases, complex join queries that would normally be slow in relational models are made to be fast and easy to construct.  

Let's take a look at the differences:

In the relational model, a many to many relationship between 2 entities is represented with an auxiliary table.  
In the example below students are mapped to courses with the enrollments table. To list all of the courses a student is enrolled in, you need to perform a join between two tables. This can be time consuming when using join operations frequently on large tables.

![sql_example](img/sql_example.png)  

---

Meanwhile using a graph database, relationships between entities become easier to visualize and query.  
Looking at the example below it is easier to see how many students are in a course, and which students are enrolled in which courses.

![graph_example](img/graph_example.png)  

---

In essence, graph databases focus on entities and the relationships between them rather than on properties of the entities themselves.

There are many graph databases implemented with [different philosophies and syntaxes](https://en.wikipedia.org/wiki/Graph_database#List_of_graph_databases). For the present course we decided to go with Neo4j and its query language Cypher as it is the [most widely used](https://db-engines.com/en/ranking/graph+dbms) and, as a consequence, one of the most mature.
A special mention to [SPARQL](https://www.ontotext.com/knowledgehub/fundamentals/what-is-sparql/) which is a type of graph database used to query the [EU open data portal](https://data.europa.eu/euodp/en/sparql) or [wikidata (which powers wikipedia)](https://www.wikidata.org/wiki/Wikidata:SPARQL_tutorial) .


We will go through each major Cypher operation. For now lets just briefly look at a comparative example with SQL extracted from [wikipedia](https://en.wikipedia.org/wiki/Graph_database#Examples):
``` SQL
SELECT p2.person_name 
FROM people p1 
JOIN friend ON (p1.person_id = friend.person_id)
JOIN people p2 ON (p2.person_id = friend.friend_id)
WHERE p1.person_name = 'Jack';
```
Becomes
``` Cypher
MATCH (p1:person)-[:FRIEND-WITH]-(p2:person)
WHERE p1.name = "Jack"
RETURN p2.name
```

Easier and more intuitive to read right?


_If anyone is not familiar with concept of [regex](https://en.wikipedia.org/wiki/Regular_expression)_